In [ ]:
# ============================================================
# EMNLP 2026 — "Doctrine Blindness in Legal AI"
# TASK B: Doctrine Reveal Experiment
# ============================================================
# SETUP CHECKLIST:
#   1. Run TaskA FIRST — Task B uses the same Drive folder
#   2. indian_bail_judgments.json must be at /content/
#   3. Same API keys in Colab Secrets as Task A
#
# WHAT THIS DOES:
#   For all 341 parity=True cases, runs 3 prompt versions:
#     V1 (Baseline):    facts only, no parity mention
#     V2 (Correct):     facts + "co-accused WAS granted bail"
#     V3 (Adversarial): facts + "co-accused was REFUSED bail"
#
#   Core metric: Doctrine Sensitivity Rate (DSR)
#     = fraction of cases where V2 and V3 predictions differ
#     = how strongly the model responds to parity doctrine
#
#   Secondary metrics:
#     - V1→V2 accuracy lift (does correct signal help?)
#     - V1→V3 accuracy drop (does wrong signal hurt?)
#     - Directional breakdown (which way do flips go?)
#
#   Ground truth throughout = real bail_outcome (no GPT-4o issue)
# ============================================================

# ══════════════════════════════════════════════════════════════
# CELL 1 — Install all required packages
# ══════════════════════════════════════════════════════════════
import sys
import subprocess

packages = ["openai", "groq", "requests", "scipy", "numpy", "pandas", "tqdm"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + packages, check=True)
print("✓ All packages installed")

# ══════════════════════════════════════════════════════════════
# CELL 2 — Mount Google Drive
# ══════════════════════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/BailBench_Doctrine'  # update to your Drive path
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f"✓ Drive mounted")
print(f"✓ Results directory: {DRIVE_DIR}")

# ══════════════════════════════════════════════════════════════
# CELL 3 — Load API keys from Colab Secrets
# ══════════════════════════════════════════════════════════════
from google.colab import userdata

def get_secret(name):
    try:
        val = userdata.get(name)
        return val if val else None
    except Exception:
        return None

OPENAI_KEY   = get_secret('OPENAI_API_KEY')
GROQ_KEYS    = [get_secret(k) for k in [
                    'GROQ_API_KEY', 'GROQ_API_KEY_2', 'GROQ_API_KEY_3',
                    'GROQ_API_KEY_4', 'GROQ_API_KEY_5']]
GROQ_KEYS    = [k for k in GROQ_KEYS if k]
MISTRAL_KEYS = [get_secret(k) for k in [
                    'MISTRAL_API_KEY', 'MISTRAL_API_KEY_2', 'MISTRAL_API_KEY_3']]
MISTRAL_KEYS = [k for k in MISTRAL_KEYS if k]

print(f"OpenAI  key  : {'✓ loaded' if OPENAI_KEY else '✗ MISSING'}")
print(f"Groq    keys : {len(GROQ_KEYS)} loaded")
print(f"Mistral keys : {len(MISTRAL_KEYS)} loaded")

assert OPENAI_KEY,        "❌ Add OPENAI_API_KEY to Colab Secrets"
assert len(GROQ_KEYS),    "❌ Add at least GROQ_API_KEY to Colab Secrets"
assert len(MISTRAL_KEYS), "❌ Add MISTRAL_API_KEY to Colab Secrets"
print("✓ All API keys loaded")

# ══════════════════════════════════════════════════════════════
# CELL 4 — All imports and global constants
# ══════════════════════════════════════════════════════════════
import json
import time
import random
import itertools
import warnings
import pandas as pd
import numpy as np
from scipy import stats as scipy_stats
from scipy.stats import norm, binomtest
from tqdm.auto import tqdm
from datetime import datetime
from openai import OpenAI
from groq import Groq
import requests

warnings.filterwarnings('ignore')

SEED        = 42
N_BOOTSTRAP = 1000
SAVE_EVERY  = 30    # save every 30 rows (3 prompts per case = save every ~10 cases)

random.seed(SEED)
np.random.seed(SEED)

_groq_cycle    = itertools.cycle(range(len(GROQ_KEYS)))
_mistral_cycle = itertools.cycle(range(len(MISTRAL_KEYS)))
_oa_client     = OpenAI(api_key=OPENAI_KEY)

print(f"✓ Imports done | SEED={SEED} | SAVE_EVERY={SAVE_EVERY} rows")

# ══════════════════════════════════════════════════════════════
# CELL 5 — Load dataset
# ══════════════════════════════════════════════════════════════
DATASET_PATH = '/content/indian_bail_judgments.json'

if not os.path.exists(DATASET_PATH):
    raise FileNotFoundError(
        "\n❌ Dataset not found at /content/indian_bail_judgments.json\n"
        "   → Upload via Colab sidebar (folder icon → upload button)"
    )

with open(DATASET_PATH) as f:
    data = json.load(f)

df = pd.DataFrame(data)
df['parity_argument_used'] = df['parity_argument_used'].astype(bool)

# Verified assertions
assert len(df) == 1200
assert df['parity_argument_used'].sum() == 341

# Task B only uses parity=True cases
parity_df = df[df['parity_argument_used'] == True].copy().reset_index(drop=True)
assert len(parity_df) == 341

grant_rate = (parity_df['bail_outcome'] == 'Granted').mean() * 100
print(f"✓ Dataset loaded")
print(f"  Task B dataset    : {len(parity_df)} parity=True cases")
print(f"  True grant rate   : {grant_rate:.1f}%  (should be 77.1%)")

# ══════════════════════════════════════════════════════════════
# CELL 6 — Three prompt builders (V1 / V2 / V3)
# ══════════════════════════════════════════════════════════════

# Exact doctrine note strings — used in all analysis labels
DOCTRINE_NOTE_V2 = "Note: The co-accused in this matter was granted bail by the court."
DOCTRINE_NOTE_V3 = "Note: The co-accused in this matter was refused bail by the court."

def _base_details(row):
    """
    Shared case details block (identical across V1/V2/V3).
    FIX 3: legal_issues removed — must match Task A prompt exactly
    so V1 results are directly comparable to Task A baseline.
    """
    return (
        f"- Court: {row.get('court', 'Unknown')}\n"
        f"- Crime Type: {row.get('crime_type', 'Unknown')}\n"
        f"- Bail Type: {row.get('bail_type', 'Unknown')}\n"
        f"- Prior Cases: {row.get('prior_cases', 'Unknown')}\n"
        f"- Case Facts: {row.get('facts', '')}"
    )

_INSTRUCTION = (
    "\n\nInstructions:\n"
    "- Reply with exactly one word: GRANTED or REJECTED\n"
    "- Do not explain your answer\n\n"
    "Your prediction:"
)

def build_v1(row):
    """Baseline — no parity doctrine information."""
    return (
        "You are analyzing an Indian court bail case.\n"
        "Based only on the information provided, predict whether bail was "
        "GRANTED or REJECTED.\n\n"
        "Case Details:\n"
        + _base_details(row) + _INSTRUCTION
    )

def build_v2(row):
    """Correct signal — co-accused was granted bail."""
    return (
        "You are analyzing an Indian court bail case.\n"
        "Based only on the information provided, predict whether bail was "
        "GRANTED or REJECTED.\n\n"
        f"{DOCTRINE_NOTE_V2}\n\n"
        "Case Details:\n"
        + _base_details(row) + _INSTRUCTION
    )

def build_v3(row):
    """Adversarial signal — co-accused was refused bail."""
    return (
        "You are analyzing an Indian court bail case.\n"
        "Based only on the information provided, predict whether bail was "
        "GRANTED or REJECTED.\n\n"
        f"{DOCTRINE_NOTE_V3}\n\n"
        "Case Details:\n"
        + _base_details(row) + _INSTRUCTION
    )

PROMPT_BUILDERS = {"V1": build_v1, "V2": build_v2, "V3": build_v3}
print("✓ Three prompt variants defined: V1 (baseline), V2 (correct), V3 (adversarial)")

# ══════════════════════════════════════════════════════════════
# CELL 7 — API call functions (same as Task A)
# ══════════════════════════════════════════════════════════════
def _parse(raw):
    t = str(raw).strip().upper()
    if t in ('GRANTED', 'REJECTED'):
        return t
    if 'GRANT' in t:
        return 'GRANTED'
    if 'REJECT' in t:
        return 'REJECTED'
    return 'INVALID'

def _parse_groq_output(raw):
    """
    Extended parser for Qwen-3-32B chain-of-thought outputs.
    Finds the LAST occurrence of GRANTED/REJECTED in the full response.
    """
    text = str(raw).strip().upper()
    if text in ('GRANTED', 'REJECTED'):
        return text
    last_g = text.rfind('GRANTED')
    last_r = text.rfind('REJECTED')
    if last_g == -1 and last_r == -1:
        return 'INVALID'
    if last_g > last_r:
        return 'GRANTED'
    if last_r > last_g:
        return 'REJECTED'
    return 'INVALID'

def call_openai(prompt, model_id, retries=4):
    for attempt in range(retries):
        try:
            r = _oa_client.chat.completions.create(
                model=model_id,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=5, temperature=0, seed=SEED)
            time.sleep(0.5)
            return _parse(r.choices[0].message.content)
        except Exception as e:
            wait = 20 * (attempt + 1)
            print(f"  [OpenAI/{model_id}] attempt {attempt+1}: {e} — wait {wait}s")
            time.sleep(wait)
    return 'ERROR'

def call_groq(prompt, model_id, retries=5):
    """Standard Groq call for Llama-4-Scout and Llama-3.3-70B."""
    for attempt in range(retries):
        ki = next(_groq_cycle)
        try:
            r = Groq(api_key=GROQ_KEYS[ki]).chat.completions.create(
                model=model_id,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=10, temperature=0)
            time.sleep(2.5)
            return _parse(r.choices[0].message.content)
        except Exception as e:
            wait = 30*(attempt+1) if ('429' in str(e) or 'rate' in str(e).lower()) else 10
            print(f"  [Groq key={ki}] attempt {attempt+1}: {e} — wait {wait}s")
            time.sleep(wait)
    return 'ERROR'

def call_groq_qwen(prompt, retries=5):
    """
    Groq call for Qwen-3-32B.
    Uses max_tokens=512 + _parse_groq_output to handle chain-of-thought.
    extra_body thinking param is NOT supported on Groq's Qwen endpoint.
    """
    model_id = "qwen/qwen3-32b"
    for attempt in range(retries):
        ki = next(_groq_cycle)
        try:
            r = Groq(api_key=GROQ_KEYS[ki]).chat.completions.create(
                model=model_id,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=512, temperature=0,
            )
            time.sleep(2.5)
            return _parse_groq_output(r.choices[0].message.content)
        except Exception as e:
            wait = 30*(attempt+1) if ('429' in str(e) or 'rate' in str(e).lower()) else 10
            print(f"  [Groq/Qwen key={ki}] attempt {attempt+1}: {e} — wait {wait}s")
            time.sleep(wait)
    return 'ERROR'

def call_mistral(prompt, model_id, retries=4):
    for attempt in range(retries):
        ki = next(_mistral_cycle)
        try:
            r = requests.post(
                "https://api.mistral.ai/v1/chat/completions",
                headers={"Authorization": f"Bearer {MISTRAL_KEYS[ki]}",
                         "Content-Type": "application/json"},
                json={"model": model_id,
                      "messages": [{"role": "user", "content": prompt}],
                      "max_tokens": 5, "temperature": 0, "random_seed": SEED},
                timeout=30)
            r.raise_for_status()
            time.sleep(1.5)
            return _parse(r.json()['choices'][0]['message']['content'])
        except Exception as e:
            wait = 45*(attempt+1) if '429' in str(e) else 10
            print(f"  [Mistral key={ki}] attempt {attempt+1}: {e} — wait {wait}s")
            time.sleep(wait)
    return 'ERROR'

print("✓ API call functions defined")

# ══════════════════════════════════════════════════════════════
# CELL 8 — Model registry
# ══════════════════════════════════════════════════════════════
MODELS = [
    {"name": "GPT-4o",        "provider": "openai",
     "fn": lambda p: call_openai(p, "gpt-4o")},
    {"name": "GPT-4o-mini",   "provider": "openai",
     "fn": lambda p: call_openai(p, "gpt-4o-mini")},
    {"name": "Llama-4-Scout", "provider": "groq",
     "fn": lambda p: call_groq(p, "meta-llama/llama-4-scout-17b-16e-instruct")},
    {"name": "Llama-3.3-70B", "provider": "groq",
     "fn": lambda p: call_groq(p, "llama-3.3-70b-versatile")},
    {"name": "Qwen-3-32B",    "provider": "groq",
     "fn": lambda p: call_groq_qwen(p)},           # FIX 2: thinking disabled
    {"name": "Mistral-Large", "provider": "mistral",
     "fn": lambda p: call_mistral(p, "mistral-large-latest")},
]

total_calls = len(parity_df) * 3 * len(MODELS)
print(f"✓ {len(MODELS)} models registered")
print(f"  341 cases × 3 prompts × {len(MODELS)} models = {total_calls} total API calls")

# ══════════════════════════════════════════════════════════════
# CELL 9 — Experiment runner (with auto-resume)
# ══════════════════════════════════════════════════════════════
def run_task_b(model_cfg, parity_df):
    """
    For each parity=True case, runs V1, V2, V3 prompts.
    Output: one row per (case_id, version) — 3 rows per case.
    Auto-resumes from Drive checkpoint on disconnect.
    """
    name      = model_cfg["name"]
    safe_name = name.replace("-", "_").replace(" ", "_")
    partial_f = f"{DRIVE_DIR}/taskB_{safe_name}_partial.csv"
    final_f   = f"{DRIVE_DIR}/taskB_{safe_name}_FINAL.csv"

    if os.path.exists(final_f):
        result = pd.read_csv(final_f)
        print(f"  ✓ {name} Task B already complete ({len(result)} rows) — loaded")
        return result

    done_keys = set()
    prior     = []
    if os.path.exists(partial_f):
        prev      = pd.read_csv(partial_f)
        done_keys = set(prev['case_id'].astype(str) + "_" + prev['version'])
        prior     = prev.to_dict('records')
        print(f"  ↩ Resuming {name}: {len(done_keys)} (case,version) pairs done")
    else:
        print(f"  → Starting {name}: {len(parity_df)*3} calls")

    call_fn = model_cfg["fn"]
    new     = []
    n_calls = 0

    print(f"\n{'='*60}")
    print(f"  Model     : {name}")
    print(f"  Cases     : {len(parity_df)}")
    print(f"  Prompts   : 3 per case (V1 baseline, V2 correct, V3 adversarial)")
    print(f"  Started   : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"{'='*60}\n")

    for _, row in tqdm(parity_df.iterrows(), total=len(parity_df), desc=name):
        cid = str(row['case_id'])

        for version, build_fn in PROMPT_BUILDERS.items():
            key = f"{cid}_{version}"
            if key in done_keys:
                continue

            pred = call_fn(build_fn(row))
            n_calls += 1

            new.append({
                "case_id":        cid,
                "model":          name,
                "version":        version,          # V1 / V2 / V3
                "bail_outcome":   str(row['bail_outcome']),
                "crime_type":     str(row.get('crime_type', 'Unknown')),
                "accused_gender": str(row.get('accused_gender', 'Unknown')),
                "accused_name":   str(row.get('accused_name', 'Unknown')),
                "bail_type":      str(row.get('bail_type', 'Unknown')),
                "court":          str(row.get('court', 'Unknown')),
                "region":         str(row.get('region', 'Unknown')),
                "prediction":     pred,
                "correct":        (pred.capitalize() == str(row['bail_outcome'])),
                "timestamp":      datetime.now().isoformat(),
            })

            if n_calls % SAVE_EVERY == 0:
                combined = pd.DataFrame(prior + new)
                combined.to_csv(partial_f, index=False)
                pct = (len(done_keys) + n_calls) / (len(parity_df)*3) * 100
                print(f"  💾 {len(combined)} rows saved [{pct:.0f}% | "
                      f"{datetime.now().strftime('%H:%M:%S')}]")

    result = pd.DataFrame(prior + new)
    result.to_csv(final_f, index=False)
    if os.path.exists(partial_f):
        os.remove(partial_f)

    n_valid = result['prediction'].isin(['GRANTED','REJECTED']).sum()
    print(f"\n  ✓ {name} COMPLETE")
    print(f"    Total rows    : {len(result)}")
    print(f"    Valid preds   : {n_valid}")
    print(f"    Saved to      : {final_f}")
    return result

print("✓ Experiment runner defined")

# ══════════════════════════════════════════════════════════════
# CELL 10 — RUN ALL MODELS
# Expected runtime: ~2 hours total (341 × 3 × 6 models)
# ══════════════════════════════════════════════════════════════
print(f"Starting Task B: 341 parity cases × 3 prompts × {len(MODELS)} models")
print(f"Results auto-save every {SAVE_EVERY} calls to: {DRIVE_DIR}\n")

all_b = {}
for cfg in MODELS:
    t0 = time.time()
    all_b[cfg["name"]] = run_task_b(cfg, parity_df)
    elapsed = (time.time() - t0) / 60
    print(f"  ⏱  {cfg['name']} took {elapsed:.0f} min\n")

master_b = pd.concat(list(all_b.values()), ignore_index=True)
master_b_path = f"{DRIVE_DIR}/taskB_MASTER.csv"
master_b.to_csv(master_b_path, index=False)

expected = 341 * 3 * len(MODELS)
print(f"\n✓ All models complete")
print(f"  Master CSV    : {master_b_path}")
print(f"  Total rows    : {len(master_b)} (expected {expected})")

# ══════════════════════════════════════════════════════════════
# CELL 11 — Statistical analysis functions
# ══════════════════════════════════════════════════════════════
def wilson_ci(k, n, alpha=0.05):
    """Wilson score 95% CI for a proportion."""
    if n == 0:
        return 0.0, 0.0
    z   = norm.ppf(1 - alpha/2)
    ph  = k / n
    d   = 1 + z**2 / n
    ctr = (ph + z**2 / (2*n)) / d
    mar = z * np.sqrt(ph*(1-ph)/n + z**2/(4*n**2)) / d
    return max(0.0, ctr - mar), min(1.0, ctr + mar)

def binom_p(k, n, p0=0.01):
    """
    One-sided binomial test: H0: flip_rate <= p0 (1% natural variance).
    Returns p-value; reject H0 if p < 0.05.
    """
    if n == 0:
        return 1.0
    return binomtest(int(k), int(n), p0, alternative='greater').pvalue

def bootstrap_acc_ci(y_true, y_pred, n=N_BOOTSTRAP, seed=SEED):
    rng     = np.random.default_rng(seed)
    correct = np.array([t == p for t, p in zip(y_true, y_pred)], dtype=float)
    acc     = correct.mean()
    boots   = [rng.choice(correct, len(correct), replace=True).mean()
               for _ in range(n)]
    return acc, np.percentile(boots, 2.5), np.percentile(boots, 97.5)

def analyze_task_b(master_b):
    rows = []

    for model, mdf in master_b.groupby('model'):
        valid = mdf[mdf['prediction'].isin(['GRANTED','REJECTED'])].copy()
        valid['prediction'] = valid['prediction'].str.capitalize()

        # Pivot to wide: one row per case, columns = V1/V2/V3 predictions
        wide = valid.pivot_table(
            index='case_id', columns='version',
            values='prediction', aggfunc='first'
        ).reset_index()

        # Merge bail_outcome
        outcome_map = (valid[['case_id','bail_outcome']]
                       .drop_duplicates()
                       .set_index('case_id'))
        wide = wide.join(outcome_map, on='case_id')

        # Keep only triads with all 3 versions
        complete = wide.dropna(subset=['V1','V2','V3'])
        n = len(complete)
        if n == 0:
            print(f"  ⚠  {model}: no complete V1/V2/V3 triads")
            continue

        # ── DSR: V2 vs V3 flip rate (PRIMARY METRIC) ──────────
        dsr_flips = int((complete['V2'] != complete['V3']).sum())
        dsr       = dsr_flips / n
        dsr_lo, dsr_hi = wilson_ci(dsr_flips, n)
        dsr_p     = binom_p(dsr_flips, n)

        # Directional breakdown of V2/V3 flips
        v2g_v3r = int(((complete['V2']=='Granted') & (complete['V3']=='Rejected')).sum())
        v2r_v3g = int(((complete['V2']=='Rejected') & (complete['V3']=='Granted')).sum())

        # ── Accuracy per version ──────────────────────────────
        yt = complete['bail_outcome'].tolist()
        acc_v1, lo1, hi1 = bootstrap_acc_ci(yt, complete['V1'].tolist())
        acc_v2, lo2, hi2 = bootstrap_acc_ci(yt, complete['V2'].tolist())
        acc_v3, lo3, hi3 = bootstrap_acc_ci(yt, complete['V3'].tolist())

        # Lift = V2 acc - V1 acc (does correct signal help?)
        # Drop = V1 acc - V3 acc (does wrong signal hurt?)
        lift = acc_v2 - acc_v1
        drop = acc_v1 - acc_v3

        # Grant rates under each condition
        gr_v1 = (complete['V1'] == 'Granted').mean()
        gr_v2 = (complete['V2'] == 'Granted').mean()
        gr_v3 = (complete['V3'] == 'Granted').mean()

        rows.append({
            "Model":           model,
            "N_complete":      n,
            # ── DSR — PRIMARY METRIC ──
            "DSR%":            round(dsr * 100, 1),
            "DSR_flips":       dsr_flips,
            "DSR_CI_lo":       round(dsr_lo * 100, 1),
            "DSR_CI_hi":       round(dsr_hi * 100, 1),
            "DSR_p":           round(dsr_p, 4),
            "DSR_sig":         "✓" if dsr_p < 0.05 else "–",
            # ── Directional ──
            "V2Grant_V3Reject":v2g_v3r,   # correct says grant, adversarial says reject
            "V2Reject_V3Grant":v2r_v3g,
            # ── Accuracy ──
            "Acc_V1%":         round(acc_v1 * 100, 1),
            "Acc_V2%":         round(acc_v2 * 100, 1),
            "Acc_V3%":         round(acc_v3 * 100, 1),
            "V1toV2_lift_pp":  round(lift * 100, 1),   # positive = V2 helps
            "V1toV3_drop_pp":  round(drop * 100, 1),   # positive = V3 hurts
            # ── Grant rates ──
            "GR_V1%":          round(gr_v1 * 100, 1),
            "GR_V2%":          round(gr_v2 * 100, 1),  # should be higher than V1
            "GR_V3%":          round(gr_v3 * 100, 1),  # should be lower than V1
        })

    return pd.DataFrame(rows).sort_values("DSR%", ascending=False).reset_index(drop=True)

print("✓ Analysis functions defined")

# ══════════════════════════════════════════════════════════════
# CELL 12 — Run analysis and print results
# ══════════════════════════════════════════════════════════════
stats_b = analyze_task_b(master_b)
stats_b_path = f"{DRIVE_DIR}/taskB_STATISTICS.csv"
stats_b.to_csv(stats_b_path, index=False)
print(f"✓ Statistics saved: {stats_b_path}\n")

print("=" * 105)
print("TASK B — TABLE 2  (paper-ready)")
print("Doctrine Reveal Experiment | 341 parity=True cases | Ground truth = real bail outcomes")
print("=" * 105)
print(f"{'Model':<20} {'DSR%':>6} {'95%CI':>14} {'p-val':>7} {'Sig':>4} "
      f"{'AV1%':>6} {'AV2%':>6} {'AV3%':>6} {'Lift':>6} {'Drop':>6} "
      f"{'GR_V1':>7} {'GR_V2':>7} {'GR_V3':>7}")
print("-" * 105)
for _, r in stats_b.iterrows():
    ci = f"[{r['DSR_CI_lo']},{r['DSR_CI_hi']}]"
    print(
        f"{r['Model']:<20} {r['DSR%']:>6} {ci:>14} {r['DSR_p']:>7.4f} {r['DSR_sig']:>4} "
        f"{r['Acc_V1%']:>6} {r['Acc_V2%']:>6} {r['Acc_V3%']:>6} "
        f"{r['V1toV2_lift_pp']:>+6} {r['V1toV3_drop_pp']:>+6} "
        f"{r['GR_V1%']:>7} {r['GR_V2%']:>7} {r['GR_V3%']:>7}"
    )

print()
print("  DSR%  = Doctrine Sensitivity Rate = V2 vs V3 flip rate")
print("  95%CI = Wilson score confidence interval")
print("  Sig   = ✓ if p<0.05 (H0: flip rate ≤ 1%)")
print("  AV1/V2/V3% = Accuracy vs real outcomes per prompt version")
print("  Lift  = V2 acc − V1 acc  (positive = correct signal helps)")
print("  Drop  = V1 acc − V3 acc  (positive = adversarial signal hurts)")
print("  GR_V* = % cases predicted as Granted under each condition")
print(f"  True grant rate in parity cases = 77.1%")

# ══════════════════════════════════════════════════════════════
# CELL 13 — DSR by crime type (supplementary)
# ══════════════════════════════════════════════════════════════
crime_rows = []
for model, mdf in master_b.groupby('model'):
    valid = mdf[mdf['prediction'].isin(['GRANTED','REJECTED'])].copy()
    valid['prediction'] = valid['prediction'].str.capitalize()

    wide = valid.pivot_table(
        index=['case_id','crime_type'], columns='version',
        values='prediction', aggfunc='first'
    ).reset_index()
    complete = wide.dropna(subset=['V2','V3'])

    for crime, cdf in complete.groupby('crime_type'):
        flips = int((cdf['V2'] != cdf['V3']).sum())
        dsr   = flips / len(cdf) if len(cdf) else 0
        crime_rows.append({
            "Model": model, "Crime": crime,
            "N": len(cdf), "DSR%": round(dsr*100,1), "DSR_flips": flips
        })

crime_dsr    = pd.DataFrame(crime_rows)
crime_pivot  = (crime_dsr.pivot_table(index='Crime', columns='Model',
                                      values='DSR%').round(1))
print("\n── DSR% by crime type (Supplementary Table) ──")
print(crime_pivot.to_string())
crime_dsr.to_csv(f"{DRIVE_DIR}/taskB_CRIME_DSR.csv", index=False)
print(f"\n✓ Crime DSR saved: {DRIVE_DIR}/taskB_CRIME_DSR.csv")

# ══════════════════════════════════════════════════════════════
# CELL 14 — Key findings summary
# ══════════════════════════════════════════════════════════════
print("\n" + "="*70)
print("TASK B — KEY FINDINGS FOR PAPER")
print("="*70)

sig_models = stats_b[stats_b['DSR_sig'] == '✓']
print(f"  Models with significant DSR (p<0.05): "
      f"{len(sig_models)}/{len(stats_b)}")
print(f"  DSR range: {stats_b['DSR%'].min()}% – {stats_b['DSR%'].max()}%")
if len(sig_models):
    best_dsr = stats_b.iloc[0]
    print(f"  Highest DSR: {best_dsr['Model']} ({best_dsr['DSR%']}%, "
          f"CI=[{best_dsr['DSR_CI_lo']},{best_dsr['DSR_CI_hi']}], p={best_dsr['DSR_p']})")

print(f"\n  Files in {DRIVE_DIR}:")
for fname in sorted(os.listdir(DRIVE_DIR)):
    if fname.startswith('taskB'):
        fpath = f"{DRIVE_DIR}/{fname}"
        print(f"    {fname:<50} {os.path.getsize(fpath):>10,} bytes")

print("\n✓ TASK B COMPLETE — Run TaskC_Demographic_Doctrine.py next")